In [1]:
!rm -rf *.wav *.png final.mp4

In [2]:
!apt-get update -qq
!apt-get install -y espeak-ng espeak-ng-data libespeak-ng1 ffmpeg

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 35 not upgraded.
Need to get 4,526 kB of archives.
After this operation, 11.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpcaudio0 amd64 1.1-6build2 [8,956 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 espeak-ng-data amd64 1.50+dfsg-10ubuntu

In [3]:
!pip install -q torch torchaudio soundfile
!pip install -q "coqui-tts[codec]"
!pip install -q pillow moviepy requests

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 30.9 MB/s eta 0:00:00


In [4]:
import os
os.environ["PHONEMIZER_ESPEAK_PATH"] = "/usr/lib/x86_64-linux-gnu/libespeak-ng.so.1"
os.environ["ESPEAK_PATH"] = "/usr/lib/x86_64-linux-gnu/libespeak-ng.so.1"

/usr/lib/x86_64-linux-gnu/libespeak-ng.so.1.1.49
/usr/lib/x86_64-linux-gnu/libespeak-ng.so.1


In [ ]:
from TTS.api import TTS

tts = TTS(
    model_name="tts_models/en/vctk/vits",
    progress_bar=False,
    gpu=False
)
print("TTS ready")

In [5]:
from pathlib import Path
import json

SLIDE_PLAN_PATH = "slide_plan.json"
AUDIO_MAP_PATH  = "slide_audio_map.json"

slide_plan = json.loads(Path(SLIDE_PLAN_PATH).read_text(encoding="utf-8"))
audio_map  = json.loads(Path(AUDIO_MAP_PATH).read_text(encoding="utf-8"))

assert len(slide_plan["slides"]) == len(audio_map)
print("Slides:", len(slide_plan["slides"]))

PHONEMIZER_ESPEAK_PATH = /usr/lib/x86_64-linux-gnu/libespeak-ng.so.1


In [6]:
audio_files = []
SPEAKER = "p225"

for item in audio_map:
    sid  = item["slide_id"]
    text = item["spoken_text"]

    out = f"voice_{sid}.wav"
    tts.tts_to_file(text=text, file_path=out, speaker=SPEAKER)
    audio_files.append(out)

print(len(audio_files), "audio files generated")


ls: cannot access '/usr/share/espeak-ng-data': No such file or directory


In [7]:
from PIL import Image, ImageDraw, ImageFont
import requests
from io import BytesIO

W, H = 1280, 720
HEADER_H = int(0.20 * H)
FOOTER_H = int(0.20 * H)
BODY_H   = H - HEADER_H - FOOTER_H
LEFT_W   = W // 2
RIGHT_W  = W // 2
MARGIN   = int(0.04 * W)

WHITE = "#ffffff"
TEXT  = "#0f172a"
HEADER_BG = "#1e3a8a"
FOOTER_BG = "#000000"
ACCENT = "#2563eb"
DIAGRAM_BG = "#eef2ff"

def font(size, bold=False, mono=False):
    try:
        if mono:
            return ImageFont.truetype("DejaVuSansMono.ttf", size)
        if bold:
            return ImageFont.truetype("DejaVuSans-Bold.ttf", size)
        return ImageFont.truetype("DejaVuSans.ttf", size)
    except:
        return ImageFont.load_default()

def draw_diagram(draw, x, y, w, h, boxes):
    if not boxes:
        return False
    bh = int(h / (len(boxes) * 1.4))
    gap = int(bh * 0.35)
    cy = y
    f = font(int(bh * 0.38), bold=True)

    for i, b in enumerate(boxes):
        draw.rounded_rectangle(
            [x, cy, x + w, cy + bh],
            radius=18,
            fill=DIAGRAM_BG,
            outline=ACCENT,
            width=3
        )
        draw.text((x + 20, cy + bh // 2), b, font=f, fill=TEXT, anchor="lm")
        if i < len(boxes) - 1:
            draw.line([x + w // 2, cy + bh, x + w // 2, cy + bh + gap],
                      fill=ACCENT, width=3)
        cy += bh + gap
    return True

def fetch_photo(q):
    try:
        r = requests.get(f"https://source.unsplash.com/900x700/?{q}", timeout=8)
        return Image.open(BytesIO(r.content)).convert("RGB")
    except:
        return None

image_files = []

for slide in slide_plan["slides"]:
    i = slide["slide_id"]
    img = Image.new("RGB", (W, H), WHITE)
    d = ImageDraw.Draw(img)

    # HEADER
    d.rectangle([0, 0, W, HEADER_H], fill=HEADER_BG)
    d.text((MARGIN, HEADER_H * 0.28),
           slide["title"],
           font=font(int(HEADER_H * 0.45), bold=True),
           fill="#ffffff")

    # FOOTER
    d.rectangle([0, H - FOOTER_H, W, H], fill=FOOTER_BG)

    # BODY
    lx, ly = MARGIN, HEADER_H + MARGIN
    lw, lh = LEFT_W - 2 * MARGIN, BODY_H - 2 * MARGIN
    rx, ry = LEFT_W + MARGIN, HEADER_H + MARGIN
    rw, rh = RIGHT_W - 2 * MARGIN, BODY_H - 2 * MARGIN

    left = slide["left_panel_plan"]
    vs   = slide["visual_strategy"]

    rendered = False

    if vs in ["CONCEPT_DIAGRAM", "FLOW_DIAGRAM", "SYSTEM_DIAGRAM"]:
        rendered = draw_diagram(d, lx, ly, lw, lh, left["diagram_boxes"])

    elif vs == "PHOTO_REFERENCE":
        photo = fetch_photo(left["photo_query"])
        if photo:
            photo.thumbnail((lw, lh))
            img.paste(photo, (lx, ly))
            rendered = True

    elif vs == "CODE_BLOCK":
        f = font(int(lh * 0.07), mono=True)
        y = ly
        for line in left["description"].splitlines():
            d.text((lx, y), line, font=f, fill=TEXT)
            y += f.size + 12
        rendered = True

    elif vs == "MATH_FORMULA":
        d.text((lx, ly + lh // 2),
               left["math_formula"],
               font=font(int(lh * 0.12), bold=True),
               fill=TEXT,
               anchor="lm")
        rendered = True

    # RIGHT PANEL — GIST ONLY
    d.text((rx, ry + rh // 3),
           slide["right_panel_gist"],
           font=font(48),
           fill=TEXT)

    fname = f"slide_{i}.png"
    img.save(fname)
    image_files.append(fname)

print(len(image_files), "slides rendered")

XTTS loaded successfully


In [9]:
from moviepy.editor import ImageClip, AudioFileClip, CompositeAudioClip, concatenate_videoclips

bg = None
if Path("bg.mp3").exists():
    bg = AudioFileClip("bg.mp3").volumex(0.04)

clips = []

for i in range(len(image_files)):
    voice = AudioFileClip(f"voice_{i}.wav").volumex(2)

    tracks = [voice]
    if bg:
        tracks.insert(0, bg.audio_loop(duration=voice.duration))

    audio = CompositeAudioClip(tracks)

    clip = (
        ImageClip(image_files[i])
        .set_duration(voice.duration)
        .set_audio(audio)
    )
    clips.append(clip)

final = concatenate_videoclips(clips, method="compose")
final.write_videofile("final.mp4", fps=24, codec="libx264", audio_codec="aac")

print("Coursera-style tutorial video generated")

Segments: 36


In [11]:
audio_files = []

SPEAKER = "p225"

for item in audio_map:
    slide_id = item["slide_id"]
    text = item["spoken_text"]

    out = f"voice_{slide_id}.wav"
    tts.tts_to_file(
        text=text,
        file_path=out,
        speaker=SPEAKER
    )
    audio_files.append(out)

print(len(audio_files), "voice files created")


36 voice files created
